In [1]:
# import torch 

In [1]:
env_name = 'CartPole-v1'

In [2]:
import gymnasium as gym 

In [3]:
# env = gym.make(env_name, max_episode_steps=2000, render_mode="rgb_array") # , render_mode = 'human')
env = gym.make(env_name, max_episode_steps=2000,  render_mode = 'human')

In [4]:
env.observation_space

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)

In [5]:
env.action_space

Discrete(2)

In [6]:
obs_dim = env.observation_space.shape[0]

In [7]:
n_acts = env.action_space.n

### Lets make the NN which will learn our policy

import torch.nn as nn
hidden_sizes = [32]
layers = []
sizes = [obs_dim] + hidden_sizes + [n_acts]
activation = nn.Tanh
output_activation = nn.Identity
for j in range(len(sizes)-1):
    act = activation if j < len(sizes)-2 else output_activation
    layers += [nn.Linear(sizes[j], sizes[j+1]), act()]
mlp =  nn.Sequential(*layers)

In [8]:
import flax.linen as nn
import jax
hidden_sizes = [32]
layers = []
sizes  = [obs_dim] + hidden_sizes + [n_acts]
activation = nn.activation.tanh
output_activation = jax.nn.identity

for j in range(len(sizes) - 1):
    act = activation if j < len(sizes) - 2 else output_activation
    layers += [nn.Dense(sizes[j+1]), act]
mlp = nn.Sequential(layers)

In [9]:
logits_net = mlp

In [10]:
print(mlp)

Sequential(
    # attributes
    layers = [Dense(
        # attributes
        features = 32
        use_bias = True
        dtype = None
        param_dtype = float32
        precision = None
        kernel_init = init
        bias_init = zeros
        promote_dtype = promote_dtype
        dot_general = None
        dot_general_cls = None
    ), <PjitFunction of <function tanh at 0x10cf6fd80>>, Dense(
        # attributes
        features = np.int64(2)
        use_bias = True
        dtype = None
        param_dtype = float32
        precision = None
        kernel_init = init
        bias_init = zeros
        promote_dtype = promote_dtype
        dot_general = None
        dot_general_cls = None
    ), <PjitFunction of <function identity at 0x10d2298a0>>]
)


from torch.distributions.categorical import Categorical
def get_policy(obs):
    logits = logits_net(obs)
    return Categorical(logits=logits)

def get_action(obs):
    return get_policy(obs).sample().item()

def compute_loss(obs, act, weights):
    logp = get_policy(obs).log_prob(act)
    return -(logp * weights).mean()


In [11]:
from distrax import Categorical
def get_policy(params, obs):
    logits = logits_net.apply(params, obs)#  logits_net(obs)
    return Categorical(logits=logits)

@jax.jit
def _action(params, obs, key):
    return get_policy(params, obs).sample(seed=key)

def get_action(params, obs, key):
    return _action(params, obs, key).item()

def compute_loss(params, obs, act, weights):
    logp = get_policy(params, obs).log_prob(act)
    return -(logp * weights).mean()

from torch.optim import Adam
import numpy as np
lr = 1e-2
optimizer = Adam(logits_net.parameters(), lr=lr)

In [12]:
from optax import adam
import optax
import jax.numpy as jnp
lr = 1e-2
optimizer = adam(learning_rate=lr)

In [13]:
key = jax.random.PRNGKey(0x1234)
key, model_key = jax.random.split(key)
key, call_key = jax.random.split(key)
params = logits_net.init(model_key, 
                         jnp.zeros((1, obs_dim)))

opt_state = optimizer.init(params)

# losses, grads = jax.value_and_grad(compute_loss, has_aux=True)(params, 

In [17]:
render = False
def train_one_epoch(key, params, opt_state):
    batch_obs = []
    batch_acts = []
    batch_weights = []
    batch_rets = []
    batch_lens = []

    obs, _ = env.reset()
    done = False
    ep_rews = []

    finished_rendering_this_epoch = False
    while True:

        if (not finished_rendering_this_epoch) and render:
            env.render()
        batch_obs.append(obs.copy())

        # act = get_action(torch.as_tensor(obs, dtype=torch.float32))
        key, sub_key = jax.random.split(key)
        act = get_action(params, obs, sub_key)
        obs, rew, terminated, truncated, _ = env.step(act)
        done = terminated or truncated
        batch_acts.append(act)
        ep_rews.append(rew)

        if done:
            ep_ret, ep_len = sum(ep_rews), len(ep_rews)
            batch_rets.append(ep_ret)
            batch_lens.append(ep_len)

            batch_weights += [ep_ret] * ep_len
            (obs, _), done, ep_rews = env.reset(), False, []
            finished_rendering_this_epoch = True

            if len(batch_obs) > batch_size:
                break
    # optimizer.zero_grad() not needed in jax
    # batch_loss = compute_loss(obs=batch_obs,
    #                           act=batch_acts,
    #                           weights=batch_weights)
    batch_obs = jnp.stack(batch_obs).astype(jnp.float32)
    batch_acts = jnp.asarray(batch_acts, dtype=jnp.int32)
    batch_weights = jnp.asarray(batch_weights, dtype=jnp.float32)

    @jax.jit
    def update(params, opt_state, obs, acts, wts):
        loss, grads = jax.value_and_grad(compute_loss, has_aux=False)(
            params, obs, acts, wts)
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss
    # loss, grads = jax.value_and_grad(compute_loss, has_aux=False)(params, batch_obs,
    #                                     batch_acts, 
    #                                     batch_weights)
    # batch_loss.backward()
    # optimizer.step()
    # updates, opt_state = optimizer.update(grads, opt_state, params)
    # params = optax.apply_updates(params, updates)
    params, opt_state, loss = update(params, opt_state, batch_obs, batch_acts, batch_weights)
    return loss, batch_rets, batch_lens, key, params, opt_state

        

In [15]:
epochs = 100
batch_size = 5000
train_one_epoch(key, params, opt_state)

(Array(28.54727, dtype=float32),
 [51.0,
  74.0,
  31.0,
  61.0,
  28.0,
  35.0,
  21.0,
  18.0,
  42.0,
  36.0,
  13.0,
  58.0,
  23.0,
  33.0,
  63.0,
  27.0,
  26.0,
  28.0,
  14.0,
  28.0,
  31.0,
  37.0,
  35.0,
  14.0,
  44.0,
  32.0,
  30.0,
  39.0,
  31.0,
  13.0,
  21.0,
  72.0,
  37.0,
  31.0,
  35.0,
  27.0,
  22.0,
  28.0,
  52.0,
  21.0,
  24.0,
  59.0,
  26.0,
  18.0,
  25.0,
  45.0,
  39.0,
  78.0,
  27.0,
  20.0,
  50.0,
  51.0,
  28.0,
  41.0,
  28.0,
  50.0,
  51.0,
  64.0,
  33.0,
  70.0,
  54.0,
  25.0,
  15.0,
  19.0,
  45.0,
  21.0,
  32.0,
  41.0,
  42.0,
  20.0,
  83.0,
  48.0,
  36.0,
  80.0,
  89.0,
  27.0,
  30.0,
  12.0,
  28.0,
  34.0,
  15.0,
  64.0,
  29.0,
  70.0,
  13.0,
  81.0,
  29.0,
  41.0,
  45.0,
  14.0,
  30.0,
  18.0,
  22.0,
  35.0,
  14.0,
  38.0,
  31.0,
  33.0,
  36.0,
  29.0,
  40.0,
  52.0,
  53.0,
  15.0,
  30.0,
  49.0,
  28.0,
  17.0,
  23.0,
  52.0,
  45.0,
  18.0,
  28.0,
  28.0,
  30.0,
  37.0,
  43.0,
  25.0,
  29.0,
  20.0,
  30.0,

In [20]:
# training loop
import numpy  as np
epochs = 100
batch_size = 500

for i in range(epochs):
    batch_loss, batch_rets, batch_lens, key, params, opt_state = train_one_epoch(key, params, opt_state)
    print('epoch: %3d \t loss: %.3f \t return: %.3f \t ep_len: %.3f' % 
        (i, batch_loss, np.mean(batch_rets), np.mean(batch_lens)))

epoch:   0 	 loss: 38.483 	 return: 83.500 	 ep_len: 83.500
epoch:   1 	 loss: 40.559 	 return: 90.833 	 ep_len: 90.833
epoch:   2 	 loss: 33.355 	 return: 66.875 	 ep_len: 66.875
epoch:   3 	 loss: 34.745 	 return: 79.571 	 ep_len: 79.571
epoch:   4 	 loss: 31.736 	 return: 68.250 	 ep_len: 68.250
epoch:   5 	 loss: 39.863 	 return: 79.714 	 ep_len: 79.714
epoch:   6 	 loss: 36.297 	 return: 74.143 	 ep_len: 74.143
epoch:   7 	 loss: 32.617 	 return: 68.875 	 ep_len: 68.875
epoch:   8 	 loss: 33.666 	 return: 71.857 	 ep_len: 71.857
epoch:   9 	 loss: 33.424 	 return: 68.875 	 ep_len: 68.875
epoch:  10 	 loss: 33.991 	 return: 72.286 	 ep_len: 72.286
epoch:  11 	 loss: 30.624 	 return: 64.375 	 ep_len: 64.375
epoch:  12 	 loss: 33.785 	 return: 71.000 	 ep_len: 71.000
epoch:  13 	 loss: 36.623 	 return: 76.000 	 ep_len: 76.000
epoch:  14 	 loss: 35.707 	 return: 74.000 	 ep_len: 74.000
epoch:  15 	 loss: 35.074 	 return: 79.286 	 ep_len: 79.286
epoch:  16 	 loss: 36.610 	 return: 76.7